In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from brief_croise_1.config.conf import DATASETS
from brief_croise_1.data import download_kaggle_dataset
from brief_croise_1.analysis import detect_anomalies, get_outliers_report_Zscore, get_outliers_report_IQR, get_numerical_corr_with_target
from brief_croise_1.visualization import plot_numeric_distributions, plot_categorical_vs_target, plot_correlation_heatmap
from brief_croise_1.preprocessing import get_X_Y, split_data_stractified, preprocess_data
from brief_croise_1.train import train_models, summarize_grids

In [ ]:
download_kaggle_dataset()

### LCSV loaded; shape, dtypes and a sample displayed

In [ ]:
df = pd.read_csv(DATASETS['raw'])

# Etape 1

### comprendre la structure générale du jeu de données (types de données, dimensions, aperçus) ;

In [ ]:
print('---------------- dataset shape -----------------')#(2000, 17)
print(df.shape )#(2000, 17)
print()
print()
print('---------- dataset colums types------------------')
print(df.dtypes)
print()
print()


In [ ]:
print('---------- dataset samples------------------')
df.head(5)

### Vérifier les valeurs manquantes, les doublons, les incohérences et les valeurs aberrantes. 

In [ ]:
print('---------- dataset number of unique ids ------------------')
print(df['Student_ID'].nunique())
print()
print()
print('---------- dataset number of duplicated row ------------------')
print(df.duplicated().sum())
print()
print()
print(df.drop(columns=['Student_ID']).duplicated().sum())

In [ ]:
print('--------------- description od dataset ------------------')
display(df.describe(include='object'))
print()
print()
display(df.describe())
print()
print()
print('--------------- number of missing value columns ------------------')
print((df.isna().sum() > 0).sum()) # 7 columns
print()
print()
print('--------------- columns that have missing values ------------------')
display(list(df.columns[df.isna().sum() > 0]))
print()
print()
print('--------------- number of missing values per column ------------------')
display(df.isna().sum()) # sleep_hours 62 - GPA 38 - social_media_screen_hours 58 - diet_quality 69 - Financial_Stress 91 - Family_Support 54 - Attendance_Rate 47

### detecting the anomalies

In [ ]:
detect_anomalies(df)

In [ ]:
IQR_res = pd.DataFrame(get_outliers_report_IQR(df))
ZSCORE_res = pd.DataFrame(get_outliers_report_Zscore(df))

display(IQR_res)
display(ZSCORE_res)

### DISTRIBUTION

In [ ]:
plot_numeric_distributions(df)

In [ ]:
plot_categorical_vs_target(df)

### Analyser les relations entre les variables explicatives et la cible.


In [ ]:
columns = df.select_dtypes(include='number').columns
skewness = pd.Series([df[col].skew() for col in columns], index=columns)
display(skewness)
display(df['Mental_Health_Risk'].value_counts()) # data is almost perfect no resampling is needed


In [ ]:
correlation = plot_correlation_heatmap(df)
correlation

In [ ]:
copy, crr , related, unrelated = get_numerical_corr_with_target(df)

display(crr)
print()
print()
print("------------------- related columns (columns associated with the target) -----------------------------")
display(related)
display(list(related.index))
print()
print()
print("------------------- unrelated columns (columns not associated with the target) -----------------------------")
display(unrelated)
display(list(unrelated.index))
display(copy)

# Etape 2 & 3

In [ ]:
x_train, x_test, y_train, y_test = split_data_stractified(*get_X_Y(df))
x_train, y_train, x_test = preprocess_data(x_train, y_train, x_test, remove_outliers=True, outliers_method='iqr', handle_social_media=True)
grids = train_models(x_train, y_train)
grids_summary = summarize_grids(grids)


In [ ]:
grids_summary.sort_values(by=['recall', 'f1'], ascending=False)

# Etape 2

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

from brief_croise_1.config.conf import TARGET_LABELS
from brief_croise_1.models import get_models_with_params
from brief_croise_1.train import train_bare_models

models , _ = get_models_with_params()
metrics, scores = train_bare_models(x_train, y_train, models['LogisticRegression'])
ConfusionMatrixDisplay(metrics['cm'], display_labels=TARGET_LABELS).plot()
plt.show()

In [ ]:
scores